In [ ]:
# 1. IMPORT DEPENDENCIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

print("System Fund: Quant Engine Initialized.")

# 2. INGEST RAW MARKET DATA
# This reads all the individual CSV files from your /data/ folder
files = glob.glob("data/*.csv")
if not files:
    # Fallback in case files are in the main directory
    files = glob.glob("*.csv")

all_trades = []
for f in files:
    df = pd.read_csv(f)
    # Identify the Homerun Engine vs. Compounding Engine
    bot_type = 'BTC_Long' if 'BTC' in f.upper() else 'Short'

    # Filter for exits only and standardize dates
    exits = df[df['Type'].str.contains('Exit', na=False)].copy()
    exits['Date and time'] = pd.to_datetime(exits['Date and time'])
    exits['Bot_Type'] = bot_type
    all_trades.append(exits)

# Create the master timeline
portfolio = pd.concat(all_trades, ignore_index=True)
start_date = pd.Timestamp('2021-03-15')
portfolio = portfolio[portfolio['Date and time'] >= start_date].copy()
portfolio = portfolio.sort_values('Date and time').reset_index(drop=True)

# 3. DEFINE THE DECOUPLED RISK SIMULATOR
def simulate_system(short_risk_pct):
    equity = 10000.0
    peak_equity = 10000.0
    max_dd = 0.0

    for _, row in portfolio.iterrows():
        # Apply the Decoupled Risk Blueprint
        if row['Bot_Type'] == 'BTC_Long':
            actual_risk_multiplier = 0.03 / 0.10  # Hard-capped 3.00% Risk for BTC Long
        else:
            actual_risk_multiplier = short_risk_pct / 0.10 # Dynamic Risk for Shorts

        trade_return = (row['Net P&L %'] / 100.0) * actual_risk_multiplier
        equity *= (1 + trade_return)

        if equity > peak_equity:
            peak_equity = equity

        drawdown = (peak_equity - equity) / peak_equity
        if drawdown > max_dd:
            max_dd = drawdown

    return max_dd, equity

# 4. OPTIMIZE FOR 10% DRAWDOWN LIMIT
def objective(short_risk):
    dd, _ = simulate_system(short_risk)
    return abs(dd - 0.10) # Algorithm targets exactly 10.00% Max DD

print("Optimizing portfolio risk matrix to target 10.00% Maximum Drawdown...")
result = minimize_scalar(objective, bounds=(0.001, 0.10), method='bounded')
optimal_short_risk = result.x

# 5. RUN FINAL AUDIT & GENERATE STATS
final_max_dd, final_equity = simulate_system(optimal_short_risk)

print("-" * 50)
print("SYSTEM FUND: 5-YEAR BACKTEST RESULTS")
print("-" * 50)
print(f"Total Trades Processed : {len(portfolio):,}")
print(f"BTC Long Risk          : 3.00%")
print(f"Optimized Short Risk   : {optimal_short_risk*100:.2f}%")
print(f"Maximum Drawdown       : {final_max_dd*100:.2f}%")
print(f"Starting Capital       : $10,000.00")
print(f"Ending Equity          : ${final_equity:,.2f}")
print(f"Total Net Return       : {((final_equity/10000)-1)*100:,.2f}%")
print("-" * 50)